In [6]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False
    print('google.colab is not available; using local filesystem paths.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "datasets": "datasets",
    "trl": "trl",
    "cairosvg": "cairosvg",
    "lxml": "lxml",
    "pandas": "pandas",
    "numpy": "numpy",
    "tqdm": "tqdm",
    "PIL": "pillow",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

print(f"Python executable: {sys.executable}")
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print(f"Installed missing packages: {missing_packages}")
else:
    print("All required packages are already installed.")


Python executable: /usr/bin/python3
All required packages are already installed.


In [8]:
import os
from pathlib import Path

if "IN_COLAB" not in globals():
    try:
        import google.colab  # noqa: F401
        IN_COLAB = True
    except ModuleNotFoundError:
        IN_COLAB = False

def looks_like_project_root(path: Path) -> bool:
    return path.exists() and ((path / ".git").exists() or (path / "README.md").exists())

WORKDIR_CANDIDATES = [
    Path.cwd(),
    Path('/content/drive/MyDrive/svg-kaggle-comptetition'),
    Path('/content/drive/MyDrive/Colab Notebooks/svg-kaggle-comptetition'),
    Path('/content/drive/MyDrive/DL Midterm'),
]
WORKDIR = next((path for path in WORKDIR_CANDIDATES if looks_like_project_root(path)), Path.cwd())
os.chdir(WORKDIR)
print(f"Working directory: {WORKDIR}")
print(f"Running in Colab: {IN_COLAB}")

Working directory: /content/drive/My Drive/svg-kaggle-comptetition
Running in Colab: True


In [4]:
!pip install -q lxml

In [9]:
import io
import re
import torch
try:
    import cairosvg
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Missing Python package 'cairosvg'. Run the package-install cell in this notebook with the active VS Code kernel."
    ) from exc
except OSError as exc:
    raise OSError(
        "cairosvg is installed, but the native Cairo library is missing. On macOS run `brew install cairo`, then restart the kernel."
    ) from exc
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path
from PIL import Image
from tqdm import tqdm
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from lxml import etree

# =========================
# CONFIG
# =========================
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ARTIFACT_PREFIX = "qwen25coder15b_canon_nosvgo_len2048"
PROJECT_ROOT = Path.cwd()
PROJECT_ROOT_CANDIDATES = [
    PROJECT_ROOT,
    Path("/content/drive/MyDrive/svg-kaggle-comptetition"),
    Path("/content/drive/MyDrive/Colab Notebooks/svg-kaggle-comptetition"),
    Path("/content/drive/MyDrive/DL Midterm"),
]
MERGED_WEIGHT_FILES = (
    "model.safetensors",
    "model.safetensors.index.json",
    "pytorch_model.bin",
    "pytorch_model.bin.index.json",
)

def has_model_weights(path: Path) -> bool:
    return path.exists() and any((path / name).exists() for name in MERGED_WEIGHT_FILES)

def first_existing_path(paths, predicate):
    for candidate in paths:
        if predicate(candidate):
            return candidate
    return None

def latest_run_artifact(run_roots, artifact_dir_name, predicate):
    for runs_root in run_roots:
        if not runs_root.exists():
            continue
        for run_dir in sorted([path for path in runs_root.iterdir() if path.is_dir()], reverse=True):
            candidate = run_dir / artifact_dir_name
            if predicate(candidate):
                return candidate
    return None

RUNS_ROOT_CANDIDATES = [root / "runs" for root in PROJECT_ROOT_CANDIDATES]
MERGED_PATH_CANDIDATES = [
    PROJECT_ROOT / f"{ARTIFACT_PREFIX}_merged",
    *(root / f"{ARTIFACT_PREFIX}_merged" for root in PROJECT_ROOT_CANDIDATES),
    PROJECT_ROOT / "svg-model-merged",
    *(root / "svg-model-merged" for root in PROJECT_ROOT_CANDIDATES),
]
MERGED_PATH = first_existing_path(MERGED_PATH_CANDIDATES, has_model_weights)
if MERGED_PATH is None:
    MERGED_PATH = latest_run_artifact(RUNS_ROOT_CANDIDATES, f"{ARTIFACT_PREFIX}_merged", has_model_weights)
if MERGED_PATH is None:
    raise FileNotFoundError("Could not find a merged model. Checked current canon artifact first, then legacy svg-model-merged.")

TEST_CSV_CANDIDATES = [
    PROJECT_ROOT / "test.csv",
    *(root / "test.csv" for root in PROJECT_ROOT_CANDIDATES),
]
TEST_CSV = str(first_existing_path(TEST_CSV_CANDIDATES, lambda path: path.exists()))
if TEST_CSV == "None":
    raise FileNotFoundError("Could not find test.csv in the expected project roots.")
SUBMISSION_CSV = "submission.csv"
DEBUG_CSV = "submission_debug.csv"

PASS1_MAX_NEW_TOKENS = 1536
PASS2_MAX_NEW_TOKENS = 2048

MAX_NEW_TOKENS = 1536
BATCH_SIZE = 128

MAX_SVG_LENGTH = 16000
MAX_PATH_COUNT = 256
RENDER_SIZE = 256

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clipPath", "mask", "linearGradient",
    "radialGradient", "stop", "text", "tspan",
    "title", "desc", "style", "pattern", "marker", "filter"
}

FALLBACK_SVG = (
    '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 256 256">'
    '<rect width="256" height="256" fill="white"/>'
    '</svg>'
)

# =========================
# LOAD MODEL
# =========================
print(f"Merged model path: {MERGED_PATH}")
print(f"Test CSV path: {TEST_CSV}")

tokenizer = AutoTokenizer.from_pretrained(MERGED_PATH)
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
)

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.padding_side = "left"

# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     device_map="auto",
#     torch_dtype="auto",
# )

# model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

model.eval()

# model.generation_config.temperature = None
# model.generation_config.top_p = None
# model.generation_config.top_k = None

print("pad:", tokenizer.pad_token, tokenizer.pad_token_id)
print("bos:", tokenizer.bos_token, tokenizer.bos_token_id)
print("eos:", tokenizer.eos_token, tokenizer.eos_token_id)
print("model pad:", model.config.pad_token_id)


# =========================
# SVG HELPERS
# =========================
def strip_namespace(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag

def extract_svg(text: str) -> str:
    text = text.strip()

    # Best case: full svg block exists
    match = re.search(r"<svg\b[^>]*>.*?</svg>", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).strip()

    # If output contains SVG: marker, keep only after that
    if "SVG:" in text:
        text = text.split("SVG:", 1)[1].strip()

    # If output contains an svg start but no closing tag, keep from <svg onward
    start = text.find("<svg")
    if start != -1:
        return text[start:].strip()

    return text

def render_svg(svg: str, size: int = RENDER_SIZE):
    try:
        png = cairosvg.svg2png(
            bytestring=svg.encode("utf-8"),
            output_width=size,
            output_height=size,
        )
        img = Image.open(io.BytesIO(png)).convert("RGB")
        return np.array(img)
    except Exception:
        return None

def validity_gate(svg: str):
    if not isinstance(svg, str) or not svg.strip():
        return 0, "svg_too_long_or_empty"

    svg = svg.strip()

    if len(svg) > MAX_SVG_LENGTH:
        return 0, "svg_too_long_or_empty"

    try:
        root = ET.fromstring(svg)
    except Exception:
        return 0, "parse_failed"

    if strip_namespace(root.tag) != "svg":
        return 0, "root_not_svg"

    path_count = 0
    for elem in root.iter():
        tag = strip_namespace(elem.tag)

        if tag not in ALLOWED_TAGS:
            return 0, f"disallowed_tag:{tag}"

        if tag == "path":
            path_count += 1

    if path_count > MAX_PATH_COUNT:
        return 0, "too_many_paths"

    if render_svg(svg) is None:
        return 0, "render_failed"

    return 1, "valid"

def repair_svg(svg: str) -> str:

    if not svg:
        return svg

    svg = svg.strip()

    # Remove anything before <svg
    start = svg.find("<svg")
    if start != -1:
        svg = svg[start:]

    if "SVG:" in svg:
        svg = svg.split("SVG:", 1)[-1].strip()

    # Remove junk after last </svg>
    if "</svg>" in svg:
        end = svg.rfind("</svg>")
        svg = svg[:end + len("</svg>")]

    # If missing closing tag → add it
    if "<svg" in svg and "</svg>" not in svg:
        svg += "</svg>"

    svg = re.sub(r"[A-Za-z0-9.\-]+$", "", svg)

    # Remove broken trailing fragments
    svg = re.sub(r"<[^>]*$", "", svg)

    return svg

def recover_svg_with_lxml(svg: str) -> str:

    if not svg or "<svg" not in svg:
        return svg
    try:
        parser = etree.XMLParser(recover=True)
        root = etree.fromstring(svg.encode("utf-8"), parser=parser)
        if root is None:
            return svg
        return etree.tostring(root, encoding="unicode")
    except Exception:
        return svg

def looks_collapsed(svg: str) -> bool:
    try:
        root = ET.fromstring(svg)
    except Exception:
        return True

    paths = [e for e in root.iter() if strip_namespace(e.tag) == "path"]
    nonempty_paths = [p for p in paths if p.attrib.get("d", "").strip()]

    if len(paths) > 0 and len(nonempty_paths) == 0:
        return True

    total_elems = sum(1 for _ in root.iter())
    if total_elems <= 1:
        return True

    return False

def accept_repaired_svg(original_svg: str, candidate_svg: str) -> bool:
    valid, _ = validity_gate(candidate_svg)
    if valid == 0:
        return False

    if looks_collapsed(candidate_svg):
        return False

    if len(candidate_svg) < 80:
        return False

    return True

def clean_svg_output(raw_text: str) -> str:
    svg = extract_svg(raw_text)

    valid, reason = validity_gate(svg)
    if valid == 1:
        return svg, "valid", svg, raw_text

    repaired = repair_svg(svg)
    valid, reason = validity_gate(repaired)
    if valid == 1:
        return repaired, "repaired", svg, raw_text

    xml = recover_svg_with_lxml(repaired)
    valid, reason = validity_gate(xml)
    if valid == 1:
        return xml, "xml", svg, raw_text

    # repaired = repair_svg(svg)
    # if accept_repaired_svg(svg, repaired):
    #     return repaired, "repaired", svg, raw_text

    # recovered = recover_svg_with_lxml(repaired)
    # if accept_repaired_svg(svg, recovered):
    #     return recovered, "xml", svg, raw_text

    return FALLBACK_SVG, "fallback", svg, raw_text

# =========================
# GENERATION
# =========================
def build_model_input(prompt: str) -> str:
    return f"Prompt: {prompt}\nSVG:\n"

# def clean_svg_output_with_reason(raw_text: str):
#     svg = extract_svg(raw_text)
#     valid, reason = validity_gate(svg)
#     if valid == 0:
#         return FALLBACK_SVG, reason, svg, raw_text
#     return svg, "valid", svg, raw_text

# def generate_svg_batch(prompts, batch_size=BATCH_SIZE):
#     results = []

#     for i in tqdm(range(0, len(prompts), batch_size)):
#         batch_prompts = prompts[i:i + batch_size]

#         inputs_text = [build_model_input(p) for p in batch_prompts]

#         inputs = tokenizer(
#             inputs_text,
#             return_tensors="pt",
#             padding=True,
#             truncation=True
#         ).to(model.device)

#         with torch.no_grad():
#             outputs = model.generate(
#                 **inputs,
#                 max_new_tokens=MAX_NEW_TOKENS,
#                 do_sample=False,
#                 pad_token_id=tokenizer.pad_token_id,
#                 eos_token_id=tokenizer.eos_token_id,
#             )

#         prompt_lengths = inputs["attention_mask"].sum(dim=1).tolist()
#         generated_only = [
#             outputs[j, int(prompt_lengths[j]):]
#             for j in range(outputs.shape[0])
#         ]

#         decoded = tokenizer.batch_decode(generated_only, skip_special_tokens=True)
#         cleaned = [clean_svg_output(x) for x in decoded]
#         results.extend(cleaned)

#     return results

#def generate_svg_batch_debug(prompts, batch_size,max_new_tokens,do_sample=False,temperature=None,top_p = None,top_k = None):
def generate_svg_batch_debug(prompts, batch_size):
    final_svgs = []
    debug_rows = []
    next_index = 0
    current_batch_size = max(1, batch_size)
    progress_bar = tqdm(total=len(prompts))

    while next_index < len(prompts):
        batch_prompts = prompts[next_index:next_index + current_batch_size]
        inputs = None
        outputs = None
        try:
            inputs_text = [build_model_input(p) for p in batch_prompts]
            inputs = tokenizer(
                inputs_text,
                return_tensors="pt",
                padding=True,
                truncation=True
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            prompt_lengths = inputs["attention_mask"].sum(dim=1).tolist()
            generated_only = [
                outputs[j, int(prompt_lengths[j]):]
                for j in range(outputs.shape[0])
            ]

            decoded = tokenizer.batch_decode(generated_only, skip_special_tokens=True)

            for prompt, raw in zip(batch_prompts, decoded):
                final_svg, reason, extracted_svg, raw_text = clean_svg_output(raw)
                final_svgs.append(final_svg)
                debug_rows.append({
                    "prompt": prompt,
                    "reason": reason,
                    "raw_text": raw_text,
                    "extracted_svg": extracted_svg,
                    "final_svg": final_svg,
                })

            next_index += len(batch_prompts)
            progress_bar.update(len(batch_prompts))
        except torch.cuda.OutOfMemoryError:
            if current_batch_size == 1:
                raise
            current_batch_size = max(1, current_batch_size // 2)
            print(f"CUDA OOM. Retrying with batch size {current_batch_size}.")
        finally:
            del inputs
            del outputs
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    progress_bar.close()
    return final_svgs, pd.DataFrame(debug_rows)

# =========================
# FINAL SUBMISSION
# =========================
def build_submission_csv(
    test_csv_path: str = TEST_CSV,
    output_csv_path: str = SUBMISSION_CSV,
    debug_csv_path: str = DEBUG_CSV,
    batch_size: int = BATCH_SIZE,
    # retry_mode="sample",
):
    test_df = pd.read_csv(test_csv_path)
    test_df = test_df.dropna(subset=["id", "prompt"]).copy()
    test_df["prompt"] = test_df["prompt"].astype(str)

    # sample_df = test_df.iloc[:100].copy()

    # prompts = sample_df["prompt"].tolist()
    # ids = sample_df["id"].tolist()

    prompts = test_df["prompt"].tolist()
    ids = test_df["id"].tolist()

    svgs, debug_df = generate_svg_batch_debug(prompts, batch_size=batch_size)
    print(debug_df["reason"].value_counts())

    assert len(ids) == len(svgs), f"ids={len(ids)}, svgs={len(svgs)}"

    submission_df = pd.DataFrame({
        "id": ids,
        "svg": svgs
    })

    submission_df.to_csv(output_csv_path, index=False)
    debug_df.to_csv("submission_debug.csv", index=False)
    return submission_df

    # print("Pass 1")
    # svgs_pass1, debug_df = generate_svg_batch_debug(
    #     prompts=prompts,
    #     batch_size=batch_size,
    #     max_new_tokens=PASS1_MAX_NEW_TOKENS,
    #     do_sample=False,
    # )

    # debug_df.insert(0, "id", ids)

    # fallback_mask = debug_df["stage"] == "fallback"
    # fallback_indices = debug_df.index[fallback_mask].tolist()

    # print("Pass 1 stage counts:")
    # print(debug_df["stage"].value_counts())

    # final_svgs = svgs_pass1[:]

    # if fallback_indices:
    #     retry_prompts = [prompts[i] for i in fallback_indices]

    #     print(f"Retrying {len(fallback_indices)} fallback rows with mode={retry_mode}")

    #     if retry_mode == "sample":
    #         retry_svgs, retry_debug = generate_svg_batch_debug(
    #             prompts=retry_prompts,
    #             batch_size=batch_size,
    #             max_new_tokens=PASS2_MAX_NEW_TOKENS,
    #             do_sample=True,
    #             temperature=0.2,
    #             top_p=0.9,
    #             top_k=20,
    #         )
    #     elif retry_mode == "long_greedy":
    #         retry_svgs, retry_debug = generate_svg_batch_debug(
    #             prompts=retry_prompts,
    #             batch_size=batch_size,
    #             max_new_tokens=PASS2_MAX_NEW_TOKENS,
    #             do_sample=False,
    #         )
    #     else:
    #         raise ValueError("retry_mode must be 'sample' or 'long_greedy'")

    #     recovered = 0
    #     for k, idx in enumerate(fallback_indices):
    #         retry_svg = retry_svgs[k]
    #         retry_stage = retry_debug.iloc[k]["stage"]
    #         retry_raw = retry_debug.iloc[k]["raw_text"]

    #         # only replace if retry escaped fallback
    #         if retry_stage != "fallback":
    #             final_svgs[idx] = retry_svg
    #             recovered += 1

    #             debug_df.at[idx, "stage"] = f"retry_{retry_stage}"
    #             debug_df.at[idx, "raw_text"] = retry_raw
    #             debug_df.at[idx, "final_svg"] = retry_svg
    #         else:
    #             debug_df.at[idx, "stage"] = "fallback_after_retry"

    #     print("Recovered fallback rows:", recovered)

    # submission_df = pd.DataFrame({
    #     "id": ids,
    #     "svg": final_svgs
    # })

    # submission_df.to_csv(output_csv_path, index=False)
    # debug_df.to_csv(debug_csv_path, index=False)

    # print("\nFinal stage counts:")
    # print(debug_df["stage"].value_counts())

    # final_validity = submission_df["svg"].apply(lambda x: validity_gate(x)[0]).mean()
    # print("\nFinal validity:", final_validity)
    # print("Saved submission:", output_csv_path)
    # print("Saved debug:", debug_csv_path)

    # return submission_df, debug_df

# =========================
# BUILD FINAL SUBMISSION
# =========================
submission_df = build_submission_csv(
    test_csv_path=TEST_CSV,
    output_csv_path=SUBMISSION_CSV,
    batch_size=BATCH_SIZE
)

# submission_df, debug_df = build_submission_csv(
#     test_csv_path="test.csv",
#     batch_size=512,
#     retry_mode="sample",
# )

Merged model path: /content/drive/My Drive/svg-kaggle-comptetition/runs/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_merged
Test CSV path: /content/drive/My Drive/svg-kaggle-comptetition/test.csv


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

pad: <|endoftext|> 151643
bos: None None
eos: <|im_end|> 151645
model pad: None


100%|██████████| 1000/1000 [08:49<00:00,  1.89it/s]


reason
valid    748
xml      252
Name: count, dtype: int64


In [10]:
COMPETITION_VIEWBOX = f"0 0 {RENDER_SIZE} {RENDER_SIZE}"
COMPETITION_SVG_OPEN = (
    f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="{COMPETITION_VIEWBOX}" '
    f'width="{RENDER_SIZE}" height="{RENDER_SIZE}">'
)
COMPETITION_FALLBACK_SVG = (
    f"{COMPETITION_SVG_OPEN}<rect width=\"{RENDER_SIZE}\" height=\"{RENDER_SIZE}\" fill=\"white\"/></svg>"
)
SVG_BODY_RE = re.compile(r"<svg\b[^>]*>(.*)</svg>", flags=re.IGNORECASE | re.DOTALL)

def ensure_competition_svg_wrapper(svg: str) -> str:
    raw_svg = "" if pd.isna(svg) else str(svg).strip()
    if not raw_svg:
        return COMPETITION_FALLBACK_SVG

    extracted_svg = extract_svg(raw_svg)
    candidate_svg = repair_svg(extracted_svg or raw_svg)

    if "<svg" not in candidate_svg.lower():
        inner_svg = candidate_svg
    else:
        wrapper_match = SVG_BODY_RE.search(candidate_svg)
        inner_svg = wrapper_match.group(1).strip() if wrapper_match else candidate_svg

    wrapped_svg = f"{COMPETITION_SVG_OPEN}{inner_svg}</svg>"
    valid, _ = validity_gate(wrapped_svg)
    if valid == 1:
        return wrapped_svg

    recovered_svg = recover_svg_with_lxml(wrapped_svg)
    valid, _ = validity_gate(recovered_svg)
    if valid == 1:
        return recovered_svg

    return COMPETITION_FALLBACK_SVG

saved_submission_df = pd.read_csv(SUBMISSION_CSV, keep_default_na=False)
normalized_svgs = [ensure_competition_svg_wrapper(svg) for svg in saved_submission_df["svg"].tolist()]
rewrapped_rows = sum(
    original != normalized
    for original, normalized in zip(saved_submission_df["svg"].tolist(), normalized_svgs)
)

saved_submission_df = saved_submission_df.copy()
saved_submission_df["svg"] = normalized_svgs
saved_submission_df.to_csv(SUBMISSION_CSV, index=False)

strict_mask = saved_submission_df["svg"].map(lambda svg: validity_gate(svg)[0] == 1)
if not strict_mask.all():
    raise AssertionError(f"Found {int((~strict_mask).sum())} invalid rows after wrapper normalization.")

print(f"Re-saved submission to: {SUBMISSION_CSV}")
print(f"Submission rows: {len(saved_submission_df)}")
print(f"Rows rewritten with the competition wrapper: {rewrapped_rows}")
display(saved_submission_df.head())


Re-saved submission to: submission.csv
Submission rows: 1000
Rows rewritten with the competition wrapper: 1000


,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,6eede943219547c22ac56085027d33cc,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
2,ea045c7a247166f061ce504d9b7ccaab,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
3,8fe82f3af89e487b31236ca829c3f071,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
4,600464e4d92c75338462271a09b3f176,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


In [16]:
from html import escape
from IPython.display import HTML, display

preview_df = pd.read_csv(SUBMISSION_CSV, keep_default_na=False)
prompt_df = pd.read_csv(TEST_CSV, keep_default_na=False)[["id", "prompt"]].copy()
preview_df["id"] = preview_df["id"].astype(str)
prompt_df["id"] = prompt_df["id"].astype(str)
preview_df = preview_df.merge(prompt_df, on="id", how="left")
sample_count = min(20, len(preview_df))
if sample_count == 0:
    raise ValueError(f"No rows found in {SUBMISSION_CSV}.")

preview_seed = int(np.random.randint(0, 1_000_000))
sampled_preview_df = preview_df.sample(n=sample_count, random_state=preview_seed).reset_index(drop=True)

cards = []
for row in sampled_preview_df.itertuples(index=False):
    cards.append(
        (
            '<div style="border:1px solid #ddd;border-radius:8px;padding:12px;background:white;">'
            f'<div style="font-family:monospace;font-size:12px;margin-bottom:8px;">id: {escape(str(row.id))}</div>'
            f'<div style="font-size:12px;line-height:1.4;margin-bottom:8px;color:#333;">prompt: {escape(str(row.prompt))}</div>'
            f'<div style="width:256px;height:256px;border:1px solid #eee;background:#fff;overflow:hidden;">{row.svg}</div>'
            '</div>'
        )
    )

grid_html = (
    '<div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(280px, 1fr));gap:12px;">'
    + ''.join(cards)
    + '</div>'
)

print(f"Previewing {sample_count} random rows from {SUBMISSION_CSV} with seed {preview_seed}")
display(HTML(grid_html))


Previewing 20 random rows from submission.csv with seed 874593
